In [2]:
!pip install wfdb pandas numpy scikit-learn matplotlib seaborn tqdm torch torchvision --quiet

In [3]:
import os
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from collections import Counter

import wfdb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, accuracy_score, hamming_loss,
    classification_report, multilabel_confusion_matrix,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [6]:
!unzip ptb.zip

Streaming output truncated to the last 5000 lines.
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19338_hr.hea  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19339_hr.dat  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19339_hr.hea  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19340_hr.dat  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19340_hr.hea  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19341_hr.dat  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19341_hr.hea  
  inflating: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/19000/19342_hr.dat  
  inflating: ptb-xl-a-large-publicly-availabl

In [7]:
# ════════════════════════════════════════════════════════════════
import os, ast, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, hamming_loss, accuracy_score,
    classification_report, multilabel_confusion_matrix,
    roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
NC = len(CLASSES)

# ── Hyperparameters ───────────────────────────────────────────────
# === CHANGE THESE FOR REAL DATA ===
DATA_PATH     = './ptb/'  # ← your path
SAMPLING_FREQ = 100          # 100 or 500
BATCH_SIZE    = 64
EPOCHS        = 60           # 60 recommended; 30 minimum
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
NB_FILTERS    = 32           # increase to 48 for higher capacity
DEPTH         = 6            # number of Inception blocks
DROPOUT       = 0.3
USE_REAL_DATA = os.path.exists(os.path.join(DATA_PATH, 'ptbxl_database.csv'))

print(f"Device : {DEVICE}")
print(f"Mode   : {'REAL PTB-XL data' if USE_REAL_DATA else 'SYNTHETIC demo data'}")

Device : cuda
Mode   : REAL PTB-XL data


In [8]:
import wfdb

def load_raw(df, sr, path):
    key = 'filename_lr' if sr == 100 else 'filename_hr'
    data = [wfdb.rdsamp(os.path.join(path, f))
            for f in tqdm(df[key], desc='Loading ECGs')]
    return np.array([sig for sig, _ in data], dtype=np.float32)

def aggregate_superclass(scp_dict, agg_df):
    labels = []
    for code in scp_dict:
        if code in agg_df.index:
            cls = agg_df.loc[code, 'diagnostic_class']
            if isinstance(cls, str):
                labels.append(cls)
    return list(set(labels))

def build_label_matrix(series, classes):
    M = np.zeros((len(series), len(classes)), dtype=np.float32)
    for i, lbls in enumerate(series):
        for l in lbls:
            if l in classes:
                M[i, classes.index(l)] = 1.0
    return M

print("Loading metadata...")
Y = pd.read_csv(os.path.join(DATA_PATH, 'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

agg_df = pd.read_csv(os.path.join(DATA_PATH, 'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

Y['superclass'] = Y.scp_codes.apply(aggregate_superclass, agg_df=agg_df)
Y = Y[Y.superclass.map(len) > 0].copy()
print(f"Valid records: {len(Y)}")

print(f"\nLoading waveforms ({SAMPLING_FREQ} Hz)...")
X_all      = load_raw(Y, SAMPLING_FREQ, DATA_PATH)
labels_all = build_label_matrix(Y['superclass'], CLASSES)


train_mask = Y.strat_fold.isin(range(1, 9))
val_mask   = Y.strat_fold == 9
test_mask  = Y.strat_fold == 10

X_train, y_train = X_all[train_mask], labels_all[train_mask]
X_val,   y_val   = X_all[val_mask],   labels_all[val_mask]
X_test,  y_test  = X_all[test_mask],  labels_all[test_mask]

# Per-lead z-score normalization (fit on train only)
mn = X_train.mean((0,1), keepdims=True)
sd = X_train.std((0,1),  keepdims=True) + 1e-8
X_train = (X_train - mn) / sd
X_val   = (X_val   - mn) / sd
X_test  = (X_test  - mn) / sd

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Label dist (train): {dict(zip(CLASSES, y_train.sum(0).astype(int)))}")

Loading metadata...
Valid records: 21388

Loading waveforms (100 Hz)...


Loading ECGs: 100%|██████████| 21388/21388 [03:30<00:00, 101.66it/s]


Train: (17084, 1000, 12)  Val: (2146, 1000, 12)  Test: (2158, 1000, 12)
Label dist (train): {'NORM': np.int64(7596), 'MI': np.int64(4379), 'STTC': np.int64(4186), 'CD': np.int64(3907), 'HYP': np.int64(2119)}


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('PTB-XL Dataset — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Class distribution
class_counts = y_train.sum(axis=0)
colors = ['#4CAF50', '#F44336', '#FF9800', '#2196F3', '#9C27B0']
axes[0].bar(CLASSES, class_counts, color=colors, alpha=0.85, edgecolor='black')
axes[0].set_title('Sample Count per Superclass', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 50, str(int(v)), ha='center', fontweight='bold')


# 3. Sample ECG waveform (first training sample, first 3 leads)
t = np.linspace(0, 10, X_train.shape[1])
lead_names = ['I', 'II', 'III']
for lead_idx in range(3):
    axes[1].plot(t, X_train[0, :, lead_idx] + lead_idx * 1.5,
                   linewidth=0.8, label=f'Lead {lead_names[lead_idx]}')
axes[1].set_title('Sample ECG Waveform (first 3 leads)', fontweight='bold')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude (mV + offset)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('ptbxl_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved → ptbxl_eda.png')

EDA plot saved → ptbxl_eda.png


In [10]:
class ECGDataset(Dataset):

    def __init__(self, X, y, aug=False):
        self.X   = torch.from_numpy(X)
        self.y   = torch.from_numpy(y)
        self.aug = aug

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        x = self.X[i].clone()           # (T, 12)
        if self.aug:
            if torch.rand(1) > 0.5:
                x = x * torch.empty(1).uniform_(0.75, 1.25)
            if torch.rand(1) > 0.5:
                x = x + torch.randn_like(x) * 0.05
            if torch.rand(1) > 0.5:
                shift = torch.randint(-50, 50, (1,)).item()
                x = torch.roll(x, shift, dims=0)
            if torch.rand(1) > 0.7:
                drop = torch.randint(0, x.shape[1], (1,)).item()
                x[:, drop] = 0.0
        return x.permute(1, 0), self.y[i]   # (12, T), label

train_loader = DataLoader(ECGDataset(X_train, y_train, aug=True),
                          BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(ECGDataset(X_val,   y_val),
                          BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(ECGDataset(X_test,  y_test),
                          BATCH_SIZE, shuffle=False, num_workers=2)

x_b, y_b = next(iter(train_loader))
print(f"Batch shape: x={x_b.shape}  y={y_b.shape}")
print(f"Loaders — train:{len(train_loader)} val:{len(val_loader)} test:{len(test_loader)} batches")

Batch shape: x=torch.Size([64, 12, 1000])  y=torch.Size([64, 5])
Loaders — train:266 val:34 test:34 batches


In [11]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation  (Hu et al. CVPR 2018)"""
    def __init__(self, c, r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(c, max(c//r, 4)), nn.ReLU(),
            nn.Linear(max(c//r, 4), c), nn.Sigmoid())
    def forward(self, x):                        # x: (B, C, T)
        return x * self.fc(x.mean(-1)).unsqueeze(-1)


class InceptionBlockSE(nn.Module):
    """1D Inception + SE"""
    def __init__(self, in_c, nb_f=32, ks=(11, 21, 41), bn_size=32):
        super().__init__()
        self.use_bn = (in_c >= 64)
        conv_in = bn_size if self.use_bn else in_c
        if self.use_bn:
            self.bottleneck = nn.Conv1d(in_c, bn_size, 1, bias=False)
        self.convs = nn.ModuleList([
            nn.Conv1d(conv_in, nb_f, k, padding=k//2, bias=False) for k in ks])
        self.mp = nn.Sequential(
            nn.MaxPool1d(3, 1, 1),
            nn.Conv1d(in_c, nb_f, 1, bias=False))
        out_c = nb_f * (len(ks) + 1)
        self.bn   = nn.BatchNorm1d(out_c)
        self.relu = nn.ReLU()
        self.se   = SEBlock(out_c)

    def forward(self, x):
        xb = self.bottleneck(x) if self.use_bn else x
        out = torch.cat([c(xb) for c in self.convs] + [self.mp(x)], dim=1)
        return self.se(self.relu(self.bn(out)))


class MHSA1D(nn.Module):

    def __init__(self, d, h=4, drop=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, dropout=drop, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)
    def forward(self, x):                        # x: (B, C, T)
        xt = x.permute(0, 2, 1)
        o, _ = self.attn(xt, xt, xt)
        return self.norm(xt + self.drop(o)).permute(0, 2, 1)


class ECGInceptionSENet(nn.Module):

    def __init__(self, in_c=12, nc=5, depth=6,
                 nb_f=32, ks=(11, 21, 41), bn_size=32, drop=0.3):
        super().__init__()
        out_c = nb_f * (len(ks) + 1)             # 32*4 = 128
        self.blocks = nn.ModuleList()
        self.res    = nn.ModuleList()

        cur = in_c
        for i in range(depth):
            if i % 3 == 0:
                group_start = cur
            self.blocks.append(InceptionBlockSE(cur, nb_f, ks, bn_size))
            cur = out_c
            if (i + 1) % 3 == 0:

                self.res.append(nn.Sequential(
                    nn.Conv1d(group_start, out_c, 1, bias=False),
                    nn.BatchNorm1d(out_c)))

        self.attn = MHSA1D(cur, h=4, drop=0.1)
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.gmp  = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(cur*2, cur), nn.GELU(), nn.Dropout(drop),
            nn.Linear(cur, nc))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        res_inp = x; ri = 0
        for i, blk in enumerate(self.blocks):
            if i % 3 == 0:
                saved_res = res_inp
            x = blk(x)
            if (i + 1) % 3 == 0 and ri < len(self.res):
                x = torch.relu(x + self.res[ri](saved_res))
                res_inp = x; ri += 1
        x = self.attn(x)
        g = torch.cat([self.gap(x).squeeze(-1),
                       self.gmp(x).squeeze(-1)], dim=1)
        return self.head(g)                    # logits (B, nc)


model = ECGInceptionSENet(
    in_c=12, nc=NC, depth=DEPTH,
    nb_f=NB_FILTERS, ks=(11,21,41), bn_size=32, drop=DROPOUT
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: ECGInceptionSENet | params: {n_params:,}")

# forward pass
with torch.no_grad():
    T_sample = X_train.shape[1]
    dummy = torch.randn(2, 12, T_sample).to(DEVICE)
    out   = model(dummy)
print(f" Forward check: {tuple(dummy.shape)} → {tuple(out.shape)}")

Model: ECGInceptionSENet | params: 588,389
 Forward check: (2, 12, 1000) → (2, 5)


In [12]:
class AsymmetricLoss(nn.Module):

    def __init__(self, gamma_pos=4.0, gamma_neg=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gp = gamma_pos; self.gn = gamma_neg
        self.clip = clip; self.eps = eps

    def forward(self, logits, y):
        p   = torch.sigmoid(logits)
        p_m = (p - self.clip).clamp(min=0)
        loss_pos = y       * torch.log(p.clamp(self.eps))    * (1 - p)**self.gp
        loss_neg = (1 - y) * torch.log((1-p_m).clamp(self.eps)) * p_m**self.gn
        return -(loss_pos + loss_neg).mean()

criterion = AsymmetricLoss(gamma_pos=4.0, gamma_neg=1.0, clip=0.05)


In [13]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr         = LR,
    steps_per_epoch= len(train_loader),
    epochs         = EPOCHS,
    pct_start      = 0.1,          # 10% warmup
    anneal_strategy= 'cos',
    div_factor     = 25.0,
    final_div_factor=1e4)

print(f"Optimizer : AdamW  lr={LR}  wd={WEIGHT_DECAY}")
print(f"Scheduler : OneCycleLR  epochs={EPOCHS}  warmup=10%")

Optimizer : AdamW  lr=0.001  wd=0.0001
Scheduler : OneCycleLR  epochs=60  warmup=10%


In [14]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += criterion(logits, y).item()
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(y.cpu().numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    try:
        auc = roc_auc_score(labels, probs, average='macro')
    except ValueError:
        auc = 0.0
    return total_loss / len(loader), auc, probs, labels



In [15]:
print(f"{'─'*65}")
print(f"  Training ECGInceptionSENet — {EPOCHS} epochs on {DEVICE}")
print(f"{'─'*65}")
print(f"{'Ep':>4}{'TrLoss':>10}{'VaLoss':>10}{'VaAUROC':>10}{'LR':>12}{'t(s)':>8}")
print(f"{'─'*65}")

best_auc    = 0.0
best_state  = None
history     = {'train_loss':[], 'val_loss':[], 'val_auc':[]}
t0          = time.time()

for ep in range(1, EPOCHS + 1):
    tl_ = train_epoch(model, train_loader, criterion, optimizer, scheduler, DEVICE)
    vl_, vauc, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(tl_)
    history['val_loss'].append(vl_)
    history['val_auc'].append(vauc)

    marker = '*' if vauc > best_auc else '  '
    if vauc > best_auc:
        best_auc   = vauc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if ep % 5 == 0 or ep == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"{ep:>4}{tl_:>10.4f}{vl_:>10.4f}{vauc:>10.4f}"
              f"{lr_now:>12.2e}{time.time()-t0:>7.1f}s {marker}")

print(f"{'─'*65}")
print(f"Traning Completed!  |  Best Val AUROC: {best_auc:.4f}")
torch.save(best_state, 'best_ecgnet.pth')
print("Model saved → best_ecgnet.pth")

─────────────────────────────────────────────────────────────────
  Training ECGInceptionSENet — 60 epochs on cuda
─────────────────────────────────────────────────────────────────
  Ep    TrLoss    VaLoss   VaAUROC          LR    t(s)
─────────────────────────────────────────────────────────────────
   1    0.1885    0.0843    0.8626    1.04e-04   77.5s *
   5    0.0671    0.0674    0.9137    9.36e-04  386.5s *
  10    0.0578    0.0642    0.9223    9.86e-04  773.5s   
  15    0.0524    0.0663    0.9209    9.33e-04 1160.8s   
  20    0.0464    0.0732    0.9193    8.43e-04 1549.8s   
  25    0.0391    0.0716    0.9218    7.24e-04 1938.9s   
  30    0.0306    0.0765    0.9189    5.87e-04 2327.9s   
  35    0.0228    0.1001    0.9156    4.42e-04 2717.0s   
  40    0.0157    0.1191    0.9127    3.02e-04 3105.9s   
  45    0.0112    0.1230    0.9141    1.79e-04 3495.6s   
  50    0.0086    0.1522    0.9115    8.22e-05 3884.2s   
  55    0.0073    0.1611    0.9106    2.10e-05 4273.2s   
  60

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ECGInceptionSENet — Training Curves', fontsize=13, fontweight='bold')

ep_range = range(1, len(history['train_loss'])+1)
axes[0].plot(ep_range, history['train_loss'], label='Train Loss', color='#E74C3C', lw=1.5)
axes[0].plot(ep_range, history['val_loss'],   label='Val Loss',   color='#3498DB', lw=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_range, history['val_auc'], label='Val AUROC', color='#2ECC71', lw=1.8)
axes[1].axhline(best_auc, color='red', linestyle='--', alpha=0.6, label=f'Best={best_auc:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUROC')
axes[1].set_title('Validation AUROC'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [17]:
model.load_state_dict(best_state)
_, _, probs, labels = evaluate(model, test_loader, criterion, DEVICE)
preds     = (probs >= 0.5).astype(int)
COLORS    = ['#4CAF50','#F44336','#FF9800','#2196F3','#9C27B0']

# ── Per-class ──────────────────────────────────────────────────────
auroc_pc = roc_auc_score(labels,  probs, average=None)
auprc_pc = average_precision_score(labels, probs, average=None)
f1_pc    = f1_score(labels, preds, average=None, zero_division=0)
prec_pc  = np.array([(preds[:,i]*labels[:,i]).sum()/(preds[:,i].sum()+1e-8) for i in range(NC)])
rec_pc   = np.array([(preds[:,i]*labels[:,i]).sum()/(labels[:,i].sum()+1e-8) for i in range(NC)])

# ── Aggregate ──────────────────────────────────────────────────────
macro_auroc = roc_auc_score(labels, probs, average='macro')
micro_auroc = roc_auc_score(labels, probs, average='micro')
macro_auprc = average_precision_score(labels, probs, average='macro')
macro_f1    = f1_score(labels, preds, average='macro',    zero_division=0)
micro_f1    = f1_score(labels, preds, average='micro',    zero_division=0)
wt_f1       = f1_score(labels, preds, average='weighted', zero_division=0)
h_loss      = hamming_loss(labels, preds)
sub_acc     = accuracy_score(labels, preds)
ml_cm       = multilabel_confusion_matrix(labels, preds)

# ── Print table ────────────────────────────────────────────────────
print(f"{'═'*68}")
print(f"     COMPLETE TEST EVALUATION  —  ECGInceptionSENet")
print(f"{'═'*68}")

print(f"\n{'Class':<8}{'AUROC':>8}{'AUPRC':>8}{'F1':>8}{'Precision':>11}{'Recall':>9}")
print('─'*52)
for i,c in enumerate(CLASSES):
    print(f"{c:<8}{auroc_pc[i]:>8.4f}{auprc_pc[i]:>8.4f}{f1_pc[i]:>8.4f}{prec_pc[i]:>11.4f}{rec_pc[i]:>9.4f}")

print(f"\n{'Metric':<26}{'Value':>8}")
print('─'*36)
for name, val in [
    ('Macro AUROC',      macro_auroc),
    ('Micro AUROC',      micro_auroc),
    ('Macro AUPRC',      macro_auprc),
    ('Macro F1',         macro_f1),
    ('Micro F1',         micro_f1),
    ('Weighted F1',      wt_f1),
    ('Hamming Loss (↓)', h_loss),
    ('Subset Accuracy',  sub_acc),
]:
    print(f"{name:<26}{val:>8.4f}")

print(f"\n{'Class':<8}{'Sensitivity':>13}{'Specificity':>13}{'PPV':>8}{'NPV':>8}")
print('─'*44)
for i,(c,cm) in enumerate(zip(CLASSES, ml_cm)):
    tn,fp,fn,tp = cm.ravel()
    print(f"{c:<8}{tp/(tp+fn+1e-8):>13.4f}{tn/(tn+fp+1e-8):>13.4f}"
          f"{tp/(tp+fp+1e-8):>8.4f}{tn/(tn+fn+1e-8):>8.4f}")

# ── Optimal threshold search ───────────────────────────────────────
print(f"\n── Optimal Threshold (F1-max) ──")
opt_t = []
for i,c in enumerate(CLASSES):
    bf,bt = 0,0.5
    for t in np.arange(0.05,0.96,0.05):
        f = f1_score(labels[:,i],(probs[:,i]>=t).astype(int),zero_division=0)
        if f>bf: bf,bt = f,t
    opt_t.append(bt); print(f"  {c}: thresh={bt:.2f}  F1={bf:.4f}")

preds_opt = np.stack([(probs[:,i]>=opt_t[i]).astype(int) for i in range(NC)],1)
opt_mf1   = f1_score(labels, preds_opt, average='macro', zero_division=0)
print(f"\n  Macro F1 (opt thresh): {opt_mf1:.4f}  |  fixed 0.5: {macro_f1:.4f}")

print(f"\n── Classification Report ──\n")
print(classification_report(labels, preds, target_names=CLASSES, zero_division=0))

════════════════════════════════════════════════════════════════════
     COMPLETE TEST EVALUATION  —  ECGInceptionSENet
════════════════════════════════════════════════════════════════════

Class      AUROC   AUPRC      F1  Precision   Recall
────────────────────────────────────────────────────
NORM      0.9396  0.9122  0.8378     0.8676   0.8100
MI        0.9171  0.8146  0.5013     0.9541   0.3400
STTC      0.9260  0.8019  0.6596     0.8443   0.5413
CD        0.9187  0.8440  0.5515     0.9845   0.3831
HYP       0.8924  0.6431  0.5753     0.6995   0.4885

Metric                       Value
────────────────────────────────────
Macro AUROC                 0.9188
Micro AUROC                 0.9158
Macro AUPRC                 0.8032
Macro F1                    0.6251
Micro F1                    0.6817
Weighted F1                 0.6628
Hamming Loss (↓)            0.1356
Subset Accuracy             0.5042

Class     Sensitivity  Specificity     PPV     NPV
─────────────────────────────────

In [18]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('ECGInceptionSENet — Evaluation Dashboard', fontsize=14, fontweight='bold')

# 1. ROC
ax = axes[0,0]
for i,(c,col) in enumerate(zip(CLASSES,COLORS)):
    fpr,tpr,_ = roc_curve(labels[:,i], probs[:,i])
    ax.plot(fpr,tpr,label=f'{c} AUC={auroc_pc[i]:.3f}',color=col,lw=2)
ax.plot([0,1],[0,1],'k--',alpha=0.4); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 2. PR
ax = axes[0,1]
for i,(c,col) in enumerate(zip(CLASSES,COLORS)):
    prec_c,rec_c,_ = precision_recall_curve(labels[:,i],probs[:,i])
    ax.plot(rec_c,prec_c,label=f'{c} AP={auprc_pc[i]:.3f}',color=col,lw=2)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 3. Per-class bars
ax = axes[0,2]
x=np.arange(NC); w=0.27
ax.bar(x-w,auroc_pc,w,label='AUROC',color='#3498DB',alpha=0.85)
ax.bar(x,  auprc_pc,w,label='AUPRC',color='#E74C3C',alpha=0.85)
ax.bar(x+w,f1_pc,   w,label='F1',   color='#2ECC71',alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylim(0,1.1); ax.set_title('Per-Class AUROC/AUPRC/F1')
ax.legend(); ax.grid(alpha=0.3,axis='y')

# 4. Prob distributions
ax = axes[1,0]
for i,(c,col) in enumerate(zip(CLASSES,COLORS)):
    ax.hist(probs[:,i],bins=30,alpha=0.5,color=col,label=c,density=True)
ax.axvline(0.5,color='k',ls='--',lw=1.5); ax.set_xlabel('Predicted Probability')
ax.set_title('Predicted Probability Distribution'); ax.legend(fontsize=8)

# 5. Confusion matrix (class 0 as example)
ax = axes[1,1]
cm0 = ml_cm[0]
sns.heatmap(cm0,annot=True,fmt='d',cmap='Blues',ax=ax,
            xticklabels=['Pred Neg','Pred Pos'],yticklabels=['True Neg','True Pos'])
tn,fp,fn,tp = cm0.ravel()
ax.set_title(f'Confusion Matrix: NORM\nSens={tp/(tp+fn+1e-8):.3f}  Spec={tn/(tn+fp+1e-8):.3f}')

# 6. Summary bars
ax = axes[1,2]
names = ['Macro\nAUROC','Micro\nAUROC','Macro\nAUPRC','Macro\nF1','Micro\nF1','Hamming\nLoss']
vals  = [macro_auroc,micro_auroc,macro_auprc,macro_f1,micro_f1,h_loss]
cols  = ['#3498DB','#2980B9','#E74C3C','#2ECC71','#27AE60','#E67E22']
bars  = ax.bar(names,vals,color=cols,alpha=0.85,edgecolor='black')
for bar,v in zip(bars,vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
            ha='center',va='bottom',fontsize=9,fontweight='bold')
ax.set_ylim(0,1.15); ax.set_title('Aggregate Metrics Summary'); ax.grid(alpha=0.3,axis='y')

plt.tight_layout()
plt.savefig('ecgnet_evaluation.png',dpi=150,bbox_inches='tight')
plt.show()
